# Data API Collector — Databricks Quick Start

A self-contained data integration sandbox that gives you **Kafka streaming, Neo4j graphs, PostgreSQL, Redis, and Elasticsearch** — all pre-wired for Databricks consumption.

**What this enables:**
- **Multi-source ingestion demos** — Structured Streaming from Kafka, JDBC reads from Postgres, Bolt queries from Neo4j, all in one stack
- **Realistic SLED data** — 6 State/Local/Higher Ed use cases with rich schemas, weighted distributions, and cross-entity relationships
- **Custom POC data** — define your own generators via JSON (Kafka, Neo4j, Postgres) without touching server code
- **End-to-end flow** — populate → stream → query → write to Delta — all covered in the example notebooks

## Example Notebooks

| Notebook | Description |
|---|---|
| **This notebook** | Quick start — verify connectivity + one test per service |
| [`examples/kafka_streaming.ipynb`](examples/kafka_streaming.ipynb) | 9 Kafka streaming use cases (core + SLED) |
| [`examples/neo4j_graph.ipynb`](examples/neo4j_graph.ipynb) | Populate and query SLED graph data + write to Delta |
| [`examples/postgres_jdbc.ipynb`](examples/postgres_jdbc.ipynb) | Populate and read SLED relational tables + write to Delta |
| [`examples/custom_kafka.ipynb`](examples/custom_kafka.ipynb) | Custom Kafka generators — IoT, clickstream, healthcare examples |
| [`examples/custom_neo4j_postgres.ipynb`](examples/custom_neo4j_postgres.ipynb) | Custom Neo4j + Postgres generators — org chart, supply chain, survey, fleet |

## Hosting & Network Access

The Data API Collector runs via Docker Compose on any machine (laptop, VM, cloud instance).
For Databricks to reach it, you need one of:

- **Same VPC/VNet** — open security group ports (10800, 9093, 15433, 7688)
- **zrok** (simplest for dev) — free, open source, supports HTTP + TCP tunnels for all services
- **Cloud VM** — deploy on EC2/GCP/Azure with public IP

**Key ports:**

| Port | Service | Protocol |
|---|---|---|
| 10800 | Caddy (API + reverse proxy) | HTTP |
| 10443 | Caddy (HTTPS) | HTTPS |
| 9093 | Kafka (external SASL/SSL via nginx) | TCP |
| 15433 | PostgreSQL (native SSL) | TCP |
| 7688 | Neo4j Bolt (TLS via nginx) | TCP |

## Prerequisites

- Data API Collector stack running (`docker compose up -d`)
- Network access from Databricks to your host
- Databricks secrets configured:

```bash
databricks secrets create-scope data-api
databricks secrets put-secret data-api api-key --string-value "YOUR_SECRET_KEY"
databricks secrets put-secret data-api kafka-user --string-value "YOUR_KAFKA_USER"
databricks secrets put-secret data-api kafka-password --string-value "YOUR_KAFKA_PASSWORD"
databricks secrets put-secret data-api neo4j-password --string-value "YOUR_NEO4J_PASSWORD"
databricks secrets put-secret data-api postgres-password --string-value "YOUR_POSTGRES_PASSWORD"
```

In [ ]:
%pip install neo4j
dbutils.library.restartPython()

In [ ]:
%run "./_config"

In [ ]:
import neo4j

---
## 1. Service Health Checks

In [ ]:
checks = {
    "PostgreSQL ORM":  "/data-sources/orm",
    "PostgreSQL SQL":  "/data-sources/raw/sql",
    "Neo4j Health":    "/data-sources/neo4j/health",
    "Neo4j Version":   "/data-sources/neo4j/version",
    "Redis":           "/redis/test",
    "Kafka Events":    "/kafka/events?limit=1",
}

for name, path in checks.items():
    try:
        r = requests.get(f"{API_URL}{path}", headers=HEADERS, timeout=10)
        status = r.json().get("status", r.status_code)
        print(f"  {name}: {status}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

---
## 2. Quick Kafka Test

In [ ]:
resp = requests.post(f"{API_URL}/kafka/generators/start", headers=HEADERS, json={
    "use_case": "fraud_detection",
    "rows_per_batch": 100,
    "batch_interval_seconds": 1.0,
    "timeout_minutes": 5,
})
print(resp.json())

In [ ]:
fraud_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("merchant_id", IntegerType()),
    StructField("amount", DecimalType(10, 2)),
    StructField("currency", StringType()),
    StructField("merchant_category", StringType()),
    StructField("payment_method", StringType()),
    StructField("ip_address", StringType()),
    StructField("device_id", StringType()),
    StructField("latitude", DecimalType(8, 5)),
    StructField("longitude", DecimalType(9, 5)),
    StructField("card_type", StringType()),
    StructField("event_timestamp", StringType()),
])

df = kafka_to_delta("streaming-fraud-transactions", fraud_schema, "quickstart_fraud")
display(df)


---
## 3. Quick Neo4j Test

In [ ]:
driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print(f"Neo4j connected: {result.single()['test']}")

    result = session.run("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count ORDER BY count DESC")
    records = [record.data() for record in result]
    if records:
        display(spark.createDataFrame(records))
    else:
        print("No nodes yet — run examples/neo4j_graph.ipynb to populate SLED data")

driver.close()

---
## 4. Quick PostgreSQL Test

In [ ]:
df = read_pg_table("kafka_event_logs")
print(f"Kafka event logs: {df.count()} rows")
display(df.limit(10))

---
## 5. Cleanup

In [ ]:
generators = requests.get(f"{API_URL}/kafka/generators", headers=HEADERS).json()
for g in generators:
    if g["status"] == "running":
        requests.post(f"{API_URL}/kafka/generators/{g['generator_id']}/stop", headers=HEADERS)
        print(f"Stopped {g['use_case']} generator {g['generator_id']}")

cleanup = requests.delete(f"{API_URL}/kafka/generators/cleanup", headers=HEADERS).json()
print(f"Cleaned up {cleanup.get('removed', 0)} generator(s)")